# 04 — Spatial Analysis: Vaccination

State-level analysis of vaccination burden, dose distribution, growth, vaccine types, age groups and AEFI. Uses the cleaned vaccination dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (12, 6)

# Change this path only if your cleaned CSV is stored elsewhere.
DATA_PATH = Path("vaccination_clean.csv")
df = pd.read_csv(DATA_PATH)

df["Updated On"] = pd.to_datetime(df["Updated On"], errors="coerce")
df = df.sort_values(["State", "Updated On"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["Updated On"].min(), "to", df["Updated On"].max())
print("States:", df["State"].nunique())


## 1. State-level cumulative vaccination

In [ ]:
state_summary = (
    df.groupby("State", as_index=False)
      .agg(
          Total_Doses=("Total Doses Administered", "max"),
          First_Dose=("First Dose Administered", "max"),
          Second_Dose=("Second Dose Administered", "max")
      )
      .sort_values("Total_Doses", ascending=False)
)

display(state_summary.head(10))

state_summary.set_index("State")["Total_Doses"].head(15).sort_values().plot(kind="barh")
plt.title("Top States by Total Doses Administered")
plt.xlabel("Total doses")
plt.tight_layout()
plt.show()


## 2. First-dose and second-dose coverage comparison

In [ ]:
state_summary["Second_Dose_Share"] = (
    state_summary["Second_Dose"] / state_summary["First_Dose"] * 100
)
state_summary["Second_to_First_Ratio"] = (
    state_summary["Second_Dose"] / state_summary["First_Dose"]
)

display(
    state_summary.sort_values("Second_Dose_Share", ascending=False)
    [["State", "First_Dose", "Second_Dose", "Second_Dose_Share"]]
    .head(15)
)

plt.scatter(state_summary["First_Dose"], state_summary["Second_Dose"])
plt.xlabel("First doses")
plt.ylabel("Second doses")
plt.title("First vs Second Doses by State")
plt.tight_layout()
plt.show()


## 3. State-wise vaccination intensity

In [ ]:
# Daily new doses are derived from the cumulative series.
df["New_Doses"] = df.groupby("State")["Total Doses Administered"].diff()

# Negative changes are revisions, not genuine negative vaccination.
df["New_Doses_Valid"] = df["New_Doses"].where(df["New_Doses"] >= 0)

state_activity = (
    df.groupby("State", as_index=False)
      .agg(
          Mean_Daily_Doses=("New_Doses_Valid", "mean"),
          Peak_Daily_Doses=("New_Doses_Valid", "max"),
          Total_Doses=("Total Doses Administered", "max")
      )
      .sort_values("Mean_Daily_Doses", ascending=False)
)

display(state_activity.head(15))


## 4. State-wise vaccination growth

In [ ]:
growth = (
    df.groupby("State")
      .agg(
          Start_Doses=("Total Doses Administered", "first"),
          End_Doses=("Total Doses Administered", "last")
      )
)
growth["Absolute_Growth"] = growth["End_Doses"] - growth["Start_Doses"]
growth["Growth_Percent"] = growth["Absolute_Growth"] / growth["Start_Doses"].replace(0, np.nan) * 100

display(growth.sort_values("Absolute_Growth", ascending=False).head(15))


## 5. Vaccine type distribution by state

In [ ]:
vaccine_cols = [
    "Covaxin (Doses Administered)",
    "CoviShield (Doses Administered)",
    "Sputnik V (Doses Administered)"
]
available = [c for c in vaccine_cols if c in df.columns]

vaccine_state = df.groupby("State")[available].max()
display(vaccine_state.sort_values(available[0], ascending=False).head(15))

vaccine_state.head(10).plot(kind="bar", stacked=True)
plt.title("Vaccine Type Distribution — Selected States")
plt.ylabel("Doses")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Age-group distribution

In [ ]:
age_cols = [
    "18-44 Years (Doses Administered)",
    "45-60 Years (Doses Administered)",
    "60+ Years (Doses Administered)"
]
age_available = [c for c in age_cols if c in df.columns]

age_state = df.groupby("State")[age_available].max()
display(age_state.sort_values(age_available[0], ascending=False).head(15))

age_state.head(10).plot(kind="bar", stacked=True)
plt.title("Age-group Vaccination Distribution")
plt.ylabel("Doses")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 7. AEFI spatial distribution

In [ ]:
if "AEFI" in df.columns:
    aefi_state = (
        df.groupby("State")["AEFI"]
        .max()
        .sort_values(ascending=False)
    )
    display(aefi_state.head(15))

    aefi_state.head(15).sort_values().plot(kind="barh")
    plt.title("States with Highest Reported AEFI Counts")
    plt.xlabel("AEFI")
    plt.tight_layout()
    plt.show()


## 8. Spatial ranking summary

In [ ]:
top = state_summary.head(10)
bottom = state_summary.tail(10)

print("Top 10 states by total doses:")
display(top)

print("Bottom 10 states by total doses:")
display(bottom)

print("\nInterpretation:")
print("- Rankings describe cumulative reported doses, not population-adjusted coverage.")
print("- For fair state comparisons, population data should be added before claiming that one state had better coverage.")
print("- Missing optional variables such as age, AEFI, or Sputnik V should not automatically be interpreted as zero.")
